In [ ]:
from sfnb_setup import setup_notebook
result = setup_notebook(config="scala_smoke_test_config.yaml")

## 1. Scala/Java Environment Setup

Initialize the JVM, Scala REPL, and `%%scala` / `%%java` cell magics.


In [ ]:
from scala_helpers import setup_scala_environment
setup_scala_environment()

## 2. Scala Execution

Validate basic Scala language features: values, string interpolation, collections.


In [ ]:
%%scala
val greeting = "Hello from Scala"
println(greeting)

val numbers = List(1, 2, 3, 4, 5)
val doubled = numbers.map(_ * 2)
println(s"Doubled: $doubled")

val sum = numbers.reduce(_ + _)
println(s"Sum of 1..5 = $sum")

if (sum == 15 && doubled == List(2, 4, 6, 8, 10)) {
  println("[PASS] Scala execution")
} else {
  println("[FAIL] Scala execution")
}

## 3. Java Execution

Validate basic Java language features via the `%%java` magic.


In [ ]:
%%java
int[] numbers = {1, 2, 3, 4, 5};
int sum = 0;
for (int n : numbers) {
    sum += n;
}
System.out.println("Sum of 1..5 = " + sum);

java.util.List<String> items = java.util.Arrays.asList("alpha", "beta", "gamma");
System.out.println("Items: " + items);

if (sum == 15) {
    System.out.println("[PASS] Java execution");
} else {
    System.out.println("[FAIL] Java execution");
}

## 4. Snowpark Scala Query

Bootstrap a Snowpark Scala session and run a query against Snowflake.


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

from scala_helpers import bootstrap_snowpark_scala
bootstrap_snowpark_scala(session)

In [ ]:
%%scala
val df = snowpark.sql("SELECT CURRENT_TIMESTAMP() AS ts, CURRENT_USER() AS usr, CURRENT_ROLE() AS role")
df.show()

val rows = df.collect()
if (rows.length > 0) {
  println("[PASS] Snowpark Scala query")
} else {
  println("[FAIL] Snowpark Scala query")
}

## 5. Variable Transfer (Scala to Python)

Transfer data between Scala and Python using the `-o` flag.


In [ ]:
%%scala -o scala_result
val scala_result = Seq(
  Map("language" -> "Scala", "version" -> scala.util.Properties.versionString),
  Map("language" -> "Java", "version" -> System.getProperty("java.version"))
)
println(s"Created ${scala_result.length} rows for transfer")

In [ ]:
if scala_result is not None and len(scala_result) == 2:
    print(f"Received {len(scala_result)} rows from Scala")
    for row in scala_result:
        print(f"  {row}")
    print("[PASS] Variable transfer")
else:
    print("[FAIL] Variable transfer")

## Summary


In [ ]:
%%scala
println("")
println("=" * 60)
println("  Scala/Java Smoke Test -- Summary")
println("=" * 60)
println(s"  Scala version:  ${scala.util.Properties.versionString}")
println(s"  Java version:   ${System.getProperty("java.version")}")
println("-" * 60)
println("  Check output above for [PASS] / [FAIL] per section")
println("=" * 60)

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.SCALA_SMOKE_TEST_RESULTS AS
        SELECT 'scala_smoke_test' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.SCALA_SMOKE_TEST_RESULTS")
except Exception:
    pass